# XAI bằng LIME + counterfactual — DT, RF, XGBoost / CICIoT2023

Theo hướng thực nghiệm trong *Toward Enhanced Attack Detection and Explanation in Intrusion Detection System-Based IoT Environment Data* (IEEE Access 2023, DOI 10.1109/ACCESS.2023.3336678): phân tích các ca dự đoán đúng/sai bằng **LIME và counterfactual**. Bài báo không công bố số perturbation hay đủ thuật toán/cấu hình counterfactual; notebook này ghi rõ thiết kế riêng, không tuyên bố tái lập nguyên trạng. Mô hình được giải thích là ba baseline riêng DT/RF/XGBoost, khác blending của bài báo.

Chạy trên **Kaggle CPU**, Add Input ba output dưới đây và bật Internet nếu cần cài LIME. Không fit lại model, không mở test, không tính global SHAP trong phiên bản này. Validation đã dùng chọn baseline; baseline test cũng đã được xem. Đây là phân tích XAI mô tả trên các ca được chọn, không phải đánh giá độc lập hiệu năng IDS.

**8 lớp:** Normal, BruteForce, DDoS, DoS, Mirai, Recon, Spoofing, Web-Based. CSV gốc có 46 đặc trưng; `*_raw.npy` sau processing có **44 cột chưa scale**, median và thứ tự cột cố định. Giữ tên `ciciot2023-do-an-xai-rf.ipynb` theo yêu cầu, nhưng giải thích đủ ba model.

| Notebook nguồn | Output cần attach và tệp sử dụng |
| --- | --- |
| `ciciot2023-do-an-processing.ipynb` | Run `20261007_164739_35f682b9`: `X_val_raw.npy`, `y_val.npy`, `feature_schema.json`, `preprocessing_config.json`; `sample_metadata_val.pkl` tùy chọn |
| `ciciot2023-do-an-pilot.ipynb` | Run `pilot_20261008_022915_ea9a88b0`: `train_1000000/X_train_raw.npy`, **`y_train.npy`, `source_row_indices.npy`**, `pilot_manifest.json` |
| `ciciot2023-do-an-models.ipynb` | Run `models_20261008_073827_d29a37c3`: toàn bộ `handoff/DT/`, `handoff/RF/`, `handoff/XGBoost/`; gốc run: `val_float32_kept_processed_indices.npy`, `val_input_audit.json`, `artifact_checksums.json` |

Kết quả: `/kaggle/working/xai_lime_counterfactual/<run_id>/`. Download toàn bộ run để đọc các ca, biểu đồ, fidelity, kết quả không tìm thấy CF, constraints, cấu hình và thời gian.


## 1. Cấu hình thực nghiệm

Mặc định chọn 6 ca dùng chung, ưu tiên một ca đúng và một ca sai cho từng lớp Normal, BruteForce, Web-Based khi pool có đủ. Thử 2 cấu hình LIME × 2 seed, mỗi lần 5.000 perturbation fit và 1.000 perturbation holdout riêng. Các ngưỡng fidelity bên dưới là quy ước khởi đầu của đồ án, chưa được coi là chuẩn chung.

Counterfactual: tìm các prototype train đúng lớp đích, đúng ngữ cảnh giao thức; dùng khoảng cách L1 sau log1p/robust scaling và phạt số feature thay đổi. Sau đó greedy restore từng **nhóm** feature. Đây là tìm kiếm gần đúng trong pool có giới hạn; không bảo đảm nghiệm gần nhất toàn cục hoặc tính khả thi khi phát lại lưu lượng thực.

Môi trường baseline: sklearn 1.6.1 / NumPy 2.1.3 / pandas 2.3.3 / joblib 1.6.0 / XGBoost 3.4.1. Nếu gặp lỗi phiên bản, cài phù hợp rồi Restart Session. LIME yêu cầu 0.2.0.1 vì cell holdout dùng hàm sinh lân cận nội bộ của phiên bản đó; không thay phiên bản âm thầm.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations
import json, sys, subprocess, warnings, time, hashlib, uuid, platform, os
import importlib.metadata as package_metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import xgboost as xgb  # necessary to reload its saved pipeline
from IPython.display import display

MODELS = ['DT','RF','XGBoost']
CLASS_NAMES = ['Normal','BruteForce','DDoS','DoS','Mirai','Recon','Spoofing','Web-Based']
RARE_IDS = [1, 7]
SEED = 42
N_CANDIDATES_PER_CLASS = 1000
LOCAL_CASE_LIMIT = 24
ANALYSIS_CASE_LIMIT = 6
TRAIN_REFERENCE_PER_CLASS = 3000
LIME_LOCAL_BACKGROUND_N = 2000
LIME_MIN_CONTEXT_BACKGROUND = 100
LIME_NUM_SAMPLES = 5000
LIME_HOLDOUT_N = 1000
LIME_SEEDS = [42,43]
LIME_CONFIGS = [
    dict(config_id='local_quartile', discretize_continuous=True, kernel_factor=0.35),
    dict(config_id='local_continuous', discretize_continuous=False, kernel_factor=0.50),
]
# Initial project acceptance rules; report continuous metrics as well.
LIME_MIN_HOLDOUT_R2 = 0.70
LIME_MAX_POINT_ERROR = 0.05
LIME_MIN_VALID_NEIGHBOR_FRACTION = 0.90
LIME_NUM_FEATURES = 10
CF_TOTAL = 2
CF_SEED_DONORS = 3
CF_RESTORE_ROUNDS = 8
CF_MIN_TARGET_PROB = 0.50
CF_MIN_TARGET_GAP = 1e-6
CF_SPARSITY_WEIGHT = 0.10
CF_MAX_CHANGED_FEATURES = None
CF_CHANGE_RTOL = 1e-5
CF_CHANGE_ATOL = 1e-6
# Conditional explanations: these fields stay exactly equal to the query.
FIXED_CONTEXT_FEATURES = ['Protocol Type','HTTP','HTTPS','DNS','SMTP','SSH','TCP','UDP','DHCP','ARP','ICMP','IPv','LLC']
VERIFY_INPUT_HASHES = True
MODEL_DIR = ''
PROCESSED_DIR = ''
PILOT_DIR = ''
EXPECTED_MODEL_RUN = 'models_20261008_073827_d29a37c3'
EXPECTED_PROCESSING_RUN = '20261007_164739_35f682b9'
EXPECTED_PILOT_RUN = 'pilot_20261008_022915_ea9a88b0'
RNG = np.random.default_rng(SEED)
TIMINGS = []
RUN_ID = 'lime_cf_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8]
OUT_DIR = Path('/kaggle/working/xai_lime_counterfactual')/RUN_ID
OUT_DIR.mkdir(parents=True, exist_ok=False)
for name in MODELS: (OUT_DIR/name).mkdir()

try:
    from lime.lime_tabular import LimeTabularExplainer
except ImportError:
    subprocess.check_call([sys.executable,'-m','pip','install','-q','lime==0.2.0.1'])
    from lime.lime_tabular import LimeTabularExplainer
if package_metadata.version('lime') != '0.2.0.1':
    raise RuntimeError('Cần lime==0.2.0.1. Cài phiên bản này và Restart Session trước khi Run All.')
assert N_CANDIDATES_PER_CLASS > 0 and LOCAL_CASE_LIMIT >= 8 and ANALYSIS_CASE_LIMIT > 0
assert TRAIN_REFERENCE_PER_CLASS > 0 and LIME_LOCAL_BACKGROUND_N >= 2
assert LIME_CONFIGS and LIME_SEEDS and LIME_NUM_SAMPLES >= 2 and LIME_HOLDOUT_N >= 2
assert CF_TOTAL > 0 and CF_SEED_DONORS >= CF_TOTAL and CF_RESTORE_ROUNDS >= 0
print('Python:',sys.version.split()[0], '| XGBoost:',xgb.__version__, '| LIME:',package_metadata.version('lime'), '| output:',OUT_DIR)


## 2. Kiểm tra nguồn và nạp ba pipeline đã fit

Giữ audit/checksum/schema của baseline. Các tệp target/source-row của pilot được dùng để lấy prototype counterfactual từ train và lưu provenance; chúng không đi vào estimator như feature.


In [ ]:
INPUT_ROOT = Path('/kaggle/input')
if not INPUT_ROOT.exists(): raise FileNotFoundError('Hãy chạy trên Kaggle và attach ba output đầu vào.')

def unique_match(pattern, description):
    found = sorted(INPUT_ROOT.glob(pattern))
    if len(found) != 1:
        raise RuntimeError(f'Cần đúng 1 {description}; tìm thấy {len(found)}: {found[:8]}. Điền đường dẫn cấu hình.')
    return found[0]

MODEL_ROOT = Path(MODEL_DIR) if MODEL_DIR else unique_match('**/handoff/RF/pipeline.joblib', 'baseline RF bundle').parent.parent.parent
SOURCE_DIR = Path(PROCESSED_DIR) if PROCESSED_DIR else unique_match('**/X_val_raw.npy', 'processed validation').parent
PILOT_ROOT = Path(PILOT_DIR) if PILOT_DIR else unique_match('**/train_1000000/X_train_raw.npy', 'pilot 1 triệu dòng').parent.parent
VAL_X_PATH = SOURCE_DIR/'X_val_raw.npy'
VAL_Y_PATH = SOURCE_DIR/'y_val.npy'
VAL_META_PATH = SOURCE_DIR/'sample_metadata_val.pkl'
VAL_KEEP_PATH = MODEL_ROOT/'val_float32_kept_processed_indices.npy'
PILOT_X_PATH = PILOT_ROOT/'train_1000000'/'X_train_raw.npy'
PILOT_MANIFEST_PATH = PILOT_X_PATH.parent/'pilot_manifest.json'
PILOT_Y_PATH = PILOT_X_PATH.parent/'y_train.npy'
PILOT_IDS_PATH = PILOT_X_PATH.parent/'source_row_indices.npy'

required = [VAL_X_PATH, VAL_Y_PATH, VAL_KEEP_PATH, PILOT_X_PATH, PILOT_MANIFEST_PATH, PILOT_Y_PATH, PILOT_IDS_PATH,
            SOURCE_DIR/'preprocessing_config.json', SOURCE_DIR/'feature_schema.json',
            MODEL_ROOT/'val_input_audit.json', MODEL_ROOT/'artifact_checksums.json']
for name in MODELS:
    required += [MODEL_ROOT/'handoff'/name/f for f in ['pipeline.joblib', 'feature_schema.json',
                 'config.json', 'label_mapping.json', 'bundle_metadata.json', 'golden_samples.npz']]
for p in required:
    if not p.is_file(): raise FileNotFoundError(p)

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def write_json(path, data):
    Path(path).write_text(json.dumps(data, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

print('Model run:', MODEL_ROOT)
print('Processing:', SOURCE_DIR)
print('Pilot:', PILOT_ROOT)


In [ ]:
PROCESSING_CONFIG = read_json(SOURCE_DIR/'preprocessing_config.json')
PROCESSING_SCHEMA = read_json(SOURCE_DIR/'feature_schema.json')
PILOT_MANIFEST = read_json(PILOT_MANIFEST_PATH)
VAL_AUDIT = read_json(MODEL_ROOT/'val_input_audit.json')
CHECKSUMS = read_json(MODEL_ROOT/'artifact_checksums.json')
FEATURES = list(PROCESSING_CONFIG['final_features'])
INPUT_FEATURES = FEATURES.copy()
assert len(FEATURES) == 44 and len(set(FEATURES)) == 44
assert PROCESSING_CONFIG['run_id'] == EXPECTED_PROCESSING_RUN
assert FEATURES == PROCESSING_SCHEMA['output_features'] == PILOT_MANIFEST['feature_names']
assert PILOT_MANIFEST['pilot_run_id'] == EXPECTED_PILOT_RUN
assert PILOT_MANIFEST['source_processing_run_id'] == EXPECTED_PROCESSING_RUN
assert PILOT_MANIFEST['class_names'] == CLASS_NAMES and PILOT_MANIFEST['status'] == 'complete'
for rel in ['val_float32_kept_processed_indices.npy', 'val_input_audit.json']:
    assert sha256_file(MODEL_ROOT/rel) == CHECKSUMS[rel], rel
if VERIFY_INPUT_HASHES:
    for p, expected in [(VAL_X_PATH, VAL_AUDIT['raw_sha256']), (VAL_Y_PATH, VAL_AUDIT['target_sha256']),
                        (PILOT_X_PATH, PILOT_MANIFEST['features']['raw']['sha256'])]:
        print('Verifying SHA256:', p.name, flush=True)
        assert sha256_file(p) == expected, f'Input hash mismatch: {p}'

for filename in ['y_train.npy','source_row_indices.npy']:
    assert sha256_file(PILOT_X_PATH.parent/filename) == PILOT_MANIFEST['checksums_sha256'][filename]
PIPELINES, ESTIMATORS, CONFIGS, BUNDLES = {}, {}, {}, {}
for name in MODELS:
    folder = MODEL_ROOT/'handoff'/name
    cfg = read_json(folder/'config.json')
    bundle = read_json(folder/'bundle_metadata.json')
    schema = read_json(folder/'feature_schema.json')
    labels = read_json(folder/'label_mapping.json')
    assert bundle['run_id'] == EXPECTED_MODEL_RUN and cfg['model'] == name
    assert cfg['source_run'] == EXPECTED_PROCESSING_RUN and cfg['pilot_run'] == EXPECTED_PILOT_RUN
    assert cfg['budget'] == 1000000 and cfg['input_view'] == 'raw'
    assert cfg['features'] == bundle['input_features_required'] == schema['output_features'] == FEATURES
    assert labels['class_names'] == cfg['class_order'] == bundle['class_order'] == CLASS_NAMES
    assert sha256_file(PILOT_MANIFEST_PATH) == cfg['train_manifest_sha256']
    assert sha256_file(folder/'pipeline.joblib') == bundle['pipeline_sha256']
    assert sha256_file(folder/'config.json') == bundle['config_sha256']
    for f in ['bundle_metadata.json', 'feature_schema.json', 'label_mapping.json', 'golden_samples.npz']:
        rel = f'handoff/{name}/{f}'
        assert sha256_file(MODEL_ROOT/rel) == CHECKSUMS[rel], rel
    packages = ['numpy', 'pandas', 'scikit-learn', 'joblib'] + (['xgboost'] if name == 'XGBoost' else [])
    for p in packages:
        actual, expected = package_metadata.version(p), bundle['versions'][p]
        if actual != expected: warnings.warn(f'{name}: {p} runtime={actual}, baseline={expected}; golden check required.')
    pipeline = joblib.load(folder/'pipeline.joblib')
    assert list(pipeline.named_steps) == ['preprocessing', 'float32', 'model']
    assert list(pipeline.named_steps['preprocessing'].feature_names_in_) == FEATURES
    model = pipeline.named_steps['model']
    assert list(model.classes_) == list(range(8))
    PIPELINES[name], ESTIMATORS[name] = pipeline, model
    CONFIGS[name], BUNDLES[name] = cfg, bundle
    print(name, cfg['experiment_id'], cfg['model_params'])

write_json(OUT_DIR/'run_status.json', dict(status='inputs_verified', run_id=RUN_ID, models=MODELS))


## 3. Validation sau purge và đối chiếu golden samples

Nạp validation bằng memory map. Kiểm tra pipeline sau reload với golden probabilities và xác nhận preprocessing tạo cùng ma trận 44 feature cho ba model. Đây là kiểm tra sẽ chạy trên Kaggle; không refit median.


In [ ]:
XVAL = np.load(VAL_X_PATH, mmap_mode='r')
YVAL = np.load(VAL_Y_PATH, mmap_mode='r')
KEPT = np.load(VAL_KEEP_PATH, mmap_mode='r')
assert XVAL.ndim == 2 and XVAL.shape[1] == len(INPUT_FEATURES)
assert len(XVAL) == len(YVAL) and np.all((KEPT >= 0) & (KEPT < len(YVAL)))
Y_EVAL = np.asarray(YVAL[KEPT], dtype=np.int64)
assert set(np.unique(Y_EVAL)).issubset(set(range(8)))
print('Raw validation:', XVAL.shape, '| retained evaluation rows:', len(KEPT))
print(pd.Series(Y_EVAL).map(dict(enumerate(CLASS_NAMES))).value_counts().reindex(CLASS_NAMES, fill_value=0).rename('support'))

assert tuple(VAL_AUDIT['shape']) == XVAL.shape
assert len(KEPT) > 0 and len(np.unique(KEPT)) == len(KEPT)
assert set(np.unique(Y_EVAL)) == set(range(8)), 'Validation phải có đủ 8 lớp.'
assert np.issubdtype(KEPT.dtype, np.integer)

# Optional source-row traceability; .npy indices remain available if pickle is incompatible.
VAL_METADATA = None
if VAL_META_PATH.is_file():
    if VERIFY_INPUT_HASHES:
        assert sha256_file(VAL_META_PATH) == VAL_AUDIT['metadata_sha256']
    try:
        VAL_METADATA = pd.read_pickle(VAL_META_PATH)
    except Exception as e:
        print('Không đọc được metadata pickle:', repr(e))
        print('Chỉ xuất processing_row_index. Để có source_row_index/original_label, dùng pandas 2.3.3 và restart.')
    if VAL_METADATA is not None:
        assert len(VAL_METADATA) == len(YVAL)
        assert {'source_row_index', 'original_label', 'class_id'}.issubset(VAL_METADATA.columns)
        assert np.array_equal(VAL_METADATA['class_id'].to_numpy(), YVAL)



In [ ]:
def raw_frame(rows):
    values = np.asarray(rows)
    if values.ndim != 2 or values.shape[1] != len(INPUT_FEATURES):
        raise ValueError(f'Expected 44 unscaled columns; got {values.shape}')
    if not np.isfinite(values).all(): raise ValueError('Processed raw array must be finite.')
    return pd.DataFrame(values, columns=INPUT_FEATURES)

def transform_raw(rows, name='DT'):
    return np.asarray(PIPELINES[name][:-1].transform(raw_frame(rows)), dtype=np.float32)

for name in MODELS:
    folder = MODEL_ROOT/'handoff'/name
    with np.load(folder/'golden_samples.npz', allow_pickle=False) as golden:
        gx = golden['X_raw_float32']
        p = PIPELINES[name].predict_proba(raw_frame(gx))
        assert np.allclose(p, golden['probabilities'], rtol=1e-6, atol=1e-7), f'{name} golden probabilities mismatch'
        rows = golden['processed_val_indices']
        assert np.array_equal(KEPT[golden['evaluation_val_indices']], rows)
        assert np.array_equal(np.asarray(XVAL[rows], dtype=np.float32), gx)
    print(name, 'golden check: PASS')

probe_eval = np.random.default_rng(SEED + 1).choice(len(KEPT), size=min(32, len(KEPT)), replace=False)
probe_raw = np.asarray(XVAL[np.asarray(KEPT[probe_eval], dtype=np.int64)])
probe_x = transform_raw(probe_raw)
for name in MODELS:
    assert np.array_equal(transform_raw(probe_raw, name), probe_x), f'{name}: preprocessing differs'
    assert np.allclose(PIPELINES[name].predict_proba(raw_frame(probe_raw)), ESTIMATORS[name].predict_proba(probe_x),
                       rtol=1e-6, atol=1e-7)
print('Shared preprocessing and estimator/pipeline parity: PASS')
del probe_raw, probe_x


## 4. Chọn ca và ghi dự đoán của từng model

Dự đoán một pool validation nhỏ đã audit; không dùng test. Cùng một ca được giải thích cho cả ba model. Loại đúng/sai được ghi theo model, không suy từ lý do chọn mẫu. Chọn tối đa 24 ca đại diện/lỗi; từ đó lấy tối đa 6 ca phân tích sâu. Các ca dùng để phát hiện khác biệt, không để tính recall/accuracy chung.


In [ ]:
candidate_eval = []
for c in range(8):
    ids = np.flatnonzero(Y_EVAL == c)
    candidate_eval.extend(RNG.choice(ids, min(N_CANDIDATES_PER_CLASS, len(ids)), replace=False).tolist())
candidate_eval = np.asarray(sorted(set(candidate_eval)), dtype=np.int64)

def load_processed(indices):
    rows = np.asarray(KEPT[np.asarray(indices, dtype=np.int64)], dtype=np.int64)
    raw = np.asarray(XVAL[rows])
    processed = transform_raw(raw)
    # Check the actual sampled rows, not just the parity probe.
    for name in MODELS:
        assert np.array_equal(transform_raw(raw, name), processed), f'{name} sampled preprocessing mismatch'
    return rows, raw, processed

candidate_rows, candidate_raw, candidate_X = load_processed(candidate_eval)
candidate_true = Y_EVAL[candidate_eval]
CANDIDATE_PROB, CANDIDATE_PRED = {}, {}
for name in MODELS:
    start = time.perf_counter()
    CANDIDATE_PROB[name] = ESTIMATORS[name].predict_proba(candidate_X)
    CANDIDATE_PRED[name] = CANDIDATE_PROB[name].argmax(axis=1)
    print(name, f'{len(candidate_eval):,} candidates predicted in {time.perf_counter()-start:.2f}s')


In [ ]:
chosen = []
chosen_ids = set()

def add_first(mask, reason, limit=1):
    for j in np.flatnonzero(mask):
        idx = int(candidate_eval[j])
        if idx in chosen_ids: continue
        if len(chosen) >= LOCAL_CASE_LIMIT: break
        chosen.append((idx, reason)); chosen_ids.add(idx)
        limit -= 1
        if limit == 0: break

all_correct = np.logical_and.reduce([CANDIDATE_PRED[n] == candidate_true for n in MODELS])
for c in range(8):
    if np.any((candidate_true == c) & all_correct):
        add_first((candidate_true == c) & all_correct, f'correct_all_{CLASS_NAMES[c]}')
    else:
        add_first(candidate_true == c, f'representative_{CLASS_NAMES[c]}')
for c in RARE_IDS:
    for name in MODELS:
        pred = CANDIDATE_PRED[name]
        add_first((candidate_true == c) & (pred != c), f'{name}_FN_{CLASS_NAMES[c]}')
    for name in MODELS:
        pred = CANDIDATE_PRED[name]
        add_first((candidate_true != c) & (pred == c), f'{name}_FP_{CLASS_NAMES[c]}')
for name in MODELS:
    add_first((candidate_true == 0) & (CANDIDATE_PRED[name] != 0), f'{name}_Normal_false_alarm')
disagree = np.any(np.column_stack([CANDIDATE_PRED[n] for n in MODELS]) != CANDIDATE_PRED[MODELS[0]][:, None], axis=1)
add_first(disagree, 'model_disagreement', LOCAL_CASE_LIMIT)
if not chosen: raise RuntimeError('No local cases available.')

local_eval = np.asarray([r[0] for r in chosen], dtype=np.int64)
local_rows, local_raw, local_X = load_processed(local_eval)
local_true = Y_EVAL[local_eval]
LOCAL_PROB = {n: ESTIMATORS[n].predict_proba(local_X) for n in MODELS}
LOCAL_PRED = {n: LOCAL_PROB[n].argmax(axis=1) for n in MODELS}
case_table = pd.DataFrame(dict(
    case_id=[f'case_{i+1:02d}' for i in range(len(local_eval))],
    selection_reason=[r[1] for r in chosen], validation_eval_index=local_eval,
    processing_row_index=local_rows, y_true_id=local_true,
    y_true=[CLASS_NAMES[c] for c in local_true]))
if VAL_METADATA is not None:
    meta = VAL_METADATA.iloc[local_rows]
    case_table['source_row_index'] = meta['source_row_index'].to_numpy()
    case_table['original_label'] = meta['original_label'].astype(str).to_numpy()
    case_table['sample_id'] = 'val:' + case_table['source_row_index'].astype(str)

CASE_TARGETS = {}
prediction_tables = []
for i, case in case_table.iterrows():
    CASE_TARGETS[case.case_id] = sorted({int(case.y_true_id)} | {int(LOCAL_PRED[n][i]) for n in MODELS})
for name in MODELS:
    table = case_table.copy()
    table['model'] = name
    table['y_pred_id'] = LOCAL_PRED[name]
    table['y_pred'] = [CLASS_NAMES[c] for c in LOCAL_PRED[name]]
    table['correct'] = table.y_true_id == table.y_pred_id
    table['case_outcome'] = [
        'correct' if t == p else ('Normal_false_alarm' if t == 0 else
                                ('missed_' + CLASS_NAMES[t] if t in RARE_IDS else 'misclassification'))
        for t, p in zip(local_true, LOCAL_PRED[name])]
    for c, label in enumerate(CLASS_NAMES): table['probability_' + label] = LOCAL_PROB[name][:, c]
    prediction_tables.append(table)
LOCAL_PREDICTIONS = pd.concat(prediction_tables, ignore_index=True)
case_table.to_csv(OUT_DIR/'shared_local_cases.csv', index=False)
LOCAL_PREDICTIONS.to_csv(OUT_DIR/'local_predictions_three_models.csv', index=False)
np.save(OUT_DIR/'shared_local_validation_eval_indices.npy', local_eval)
display(LOCAL_PREDICTIONS[['case_id','selection_reason','model','y_true','y_pred','case_outcome']])

# Prefer a fully correct and an error/disagreement example for each focus class.
analysis_indices = []
for c in [0,1,7]:
    ids = np.flatnonzero(local_true == c).tolist()
    correct = [i for i in ids if all(LOCAL_PRED[n][i] == c for n in MODELS)]
    errors = [i for i in ids if any(LOCAL_PRED[n][i] != c for n in MODELS)]
    for group in [correct, errors]:
        if group and group[0] not in analysis_indices: analysis_indices.append(group[0])
for i in range(len(case_table)):
    if len(analysis_indices) >= ANALYSIS_CASE_LIMIT: break
    if i not in analysis_indices: analysis_indices.append(i)
analysis_indices = analysis_indices[:ANALYSIS_CASE_LIMIT]
case_table.iloc[analysis_indices].to_csv(OUT_DIR/'shared_analysis_cases.csv',index=False)
np.save(OUT_DIR/'analysis_validation_eval_indices.npy', local_eval[analysis_indices])
print('Analysis cases:',case_table.iloc[analysis_indices].case_id.tolist())


## 5. Reference train, miền giá trị và ngữ cảnh cố định

Lấy tối đa 3.000 dòng **mỗi lớp** từ pilot train, không thay dữ liệu mà model đã học. Reference được cân bằng để lớp hiếm có đủ prototype; đây không phải phân bố lưu lượng tự nhiên. LIME lấy các hàng train gần query từ reference này. Khi đủ hàng cùng ngữ cảnh giao thức, ưu tiên nhóm đó; nếu không đủ, dùng hàng gần nhất toàn pool nhưng vẫn cố định ngữ cảnh của query khi sinh perturbation.

Khoảng cách retrieval/CF sử dụng log1p cho feature luôn không âm, rồi chia robust scale train. Thống kê này chỉ dùng để phân tích XAI, không được đưa làm preprocessing mới của estimator.

Ràng buộc cơ bản: finite, các feature có miền không âm/0..1/binary được kiểm tra; quan hệ `Min <= AVG <= Max` chỉ được áp dụng nếu đúng gần như toàn bộ reference (≥99,9%), và tỷ lệ này được ghi. CF còn phải nằm trong min/max reference và giữ nguyên context. Các checks này không chứng nhận đầy đủ tính khả thi của lưu lượng mạng thực.


In [ ]:
pilot_X = np.load(PILOT_X_PATH,mmap_mode='r')
pilot_y = np.load(PILOT_Y_PATH,mmap_mode='r')
pilot_ids = np.load(PILOT_IDS_PATH,mmap_mode='r')
assert len(pilot_X) == len(pilot_y) == len(pilot_ids)
assert pilot_X.shape == tuple(PILOT_MANIFEST['features']['raw']['shape'])
assert pilot_y.ndim == pilot_ids.ndim == 1
assert np.issubdtype(pilot_y.dtype, np.integer) and np.issubdtype(pilot_ids.dtype, np.integer)
assert set(np.unique(pilot_y)) == set(range(8))
ref_idx = []
rr = np.random.default_rng(SEED+100)
for c in range(8):
    ids = np.flatnonzero(pilot_y == c)
    ref_idx.extend(rr.choice(ids,min(TRAIN_REFERENCE_PER_CLASS,len(ids)),replace=False).tolist())
ref_idx = np.asarray(sorted(ref_idx),dtype=np.int64)
ref_raw = np.asarray(pilot_X[ref_idx])
REF_X = transform_raw(ref_raw)
for name in MODELS: assert np.array_equal(transform_raw(ref_raw,name),REF_X)
REF_Y = np.asarray(pilot_y[ref_idx],dtype=np.int64)
REF_SOURCE_IDS = np.asarray(pilot_ids[ref_idx],dtype=np.int64)
del ref_raw,pilot_X,pilot_y,pilot_ids
np.save(OUT_DIR/'reference_pilot_row_indices.npy',ref_idx)
np.save(OUT_DIR/'reference_source_row_indices.npy',REF_SOURCE_IDS)
pd.DataFrame({'class_name':CLASS_NAMES,'rows':np.bincount(REF_Y,minlength=8)}).to_csv(OUT_DIR/'reference_class_support.csv',index=False)

FIXED_IDX = np.asarray([FEATURES.index(f) for f in FIXED_CONTEXT_FEATURES],dtype=np.int64)
MUTABLE_IDX = np.asarray([j for j in range(44) if j not in set(FIXED_IDX)],dtype=np.int64)
LOWER = REF_X.min(axis=0).astype(np.float64)
UPPER = REF_X.max(axis=0).astype(np.float64)
NONNEGATIVE = LOWER >= 0
UNIT_INTERVAL = (LOWER >= 0) & (UPPER <= 1)
BINARY = np.array([np.all(np.isin(REF_X[:,j],[0,1])) and np.all(np.isin(local_X[:,j],[0,1])) for j in range(44)])


def distance_encode(X):
    z = np.asarray(X,dtype=np.float64).copy()
    z[...,NONNEGATIVE] = np.log1p(np.maximum(z[...,NONNEGATIVE],0))
    return z

REF_Z = distance_encode(REF_X)
q25,q75 = np.quantile(REF_Z,[.25,.75],axis=0)
DIST_SCALE = q75-q25
fallback = REF_Z.std(axis=0)
DIST_SCALE = np.where(DIST_SCALE>1e-8,DIST_SCALE,np.where(fallback>1e-8,fallback,1))


def normalized_l1(X,query):
    return np.mean(np.abs(distance_encode(X)-distance_encode(query))/DIST_SCALE,axis=-1)


min_j,avg_j,max_j = [FEATURES.index(f) for f in ['Min','AVG','Max']]
ordered = (REF_X[:,min_j] <= REF_X[:,avg_j]+1e-6) & (REF_X[:,avg_j] <= REF_X[:,max_j]+1e-6)
ENFORCE_ORDERED_SIZE = float(ordered.mean()) >= .999


def domain_valid(X,query=None,strict_reference_bounds=False):
    x = np.atleast_2d(np.asarray(X,dtype=np.float64))
    ok = np.isfinite(x).all(axis=1)
    ok &= (x[:,NONNEGATIVE] >= -1e-6).all(axis=1)
    ok &= ((x[:,UNIT_INTERVAL] >= -1e-6) & (x[:,UNIT_INTERVAL] <= 1+1e-6)).all(axis=1)
    ok &= np.isin(x[:,BINARY],[0,1]).all(axis=1)
    if ENFORCE_ORDERED_SIZE:
        tol = 1e-6*np.maximum(1,np.abs(x[:,max_j]))
        ok &= (x[:,min_j] <= x[:,avg_j]+tol) & (x[:,avg_j] <= x[:,max_j]+tol)
    if strict_reference_bounds:
        tol = 1e-6*np.maximum(1,np.maximum(np.abs(LOWER),np.abs(UPPER)))
        ok &= ((x >= LOWER-tol) & (x <= UPPER+tol)).all(axis=1)
    if query is not None:
        ok &= np.isclose(x[:,FIXED_IDX],np.asarray(query)[FIXED_IDX],rtol=0,atol=0).all(axis=1)
    return ok


def context_match(X,query):
    return np.isclose(X[:,FIXED_IDX],query[FIXED_IDX],rtol=0,atol=0).all(axis=1)

pd.DataFrame(dict(feature=FEATURES,reference_min=LOWER,reference_max=UPPER,
    nonnegative=NONNEGATIVE,unit_interval=UNIT_INTERVAL,binary=BINARY,
    fixed_context=[j in set(FIXED_IDX) for j in range(44)],distance_scale=DIST_SCALE)).to_csv(OUT_DIR/'feature_constraints.csv',index=False)
write_json(OUT_DIR/'constraint_audit.json',dict(ordered_size_reference_fraction=float(ordered.mean()),
    enforce_Min_AVG_Max=ENFORCE_ORDERED_SIZE,description='Basic feature-domain checks; no packet-level feasibility certificate'))

REF_PROB = {}
for name in MODELS:
    start=time.perf_counter()
    REF_PROB[name]=ESTIMATORS[name].predict_proba(REF_X)
    TIMINGS.append(dict(method='reference_prediction',model=name,seconds=time.perf_counter()-start,samples=len(REF_X)))
print('Reference shape:',REF_X.shape,'| ordered-size constraint:',ENFORCE_ORDERED_SIZE)


## 6. LIME với background gần query và fidelity trên holdout riêng

LIME ở đây là **giải thích có điều kiện**, giữ nguyên giao thức và các protocol indicators. Chỉ giải thích các feature còn lại; một context cố định không có trọng số LIME không có nghĩa là nó không quan trọng với model.

Với cùng case/config/seed, cả ba model và các lớp đích dùng cùng ma trận perturbation, xác nhận SHA256. Holdout sinh độc lập với fit, không phải tập test CICIoT2023. Báo train R², weighted holdout R²/MAE, sai lệch tại query, tỷ lệ perturbation hợp lệ và độ ổn định top feature qua seed. Nếu output holdout gần hằng số, R² không đủ thông tin và không gắn nhãn đáng tin.

Mỗi config đều được lưu. Một config chung cho ba model được chọn theo fidelity holdout trung bình qua các seed; lựa chọn này là khảo sát cấu hình trên các ca đã xem, chưa phải đánh giá fidelity độc lập cuối cùng.


In [ ]:
def lime_scalar(value,target):
    if isinstance(value,dict): value=value[target]
    a=np.asarray(value).ravel()
    if a.size != 1: raise ValueError('Expected one LIME target scalar')
    return float(a[0])


def reconstruct_lime_input(query,partial,work_idx):
    full=np.repeat(query[None,:],len(partial),axis=0).astype(np.float32)
    with np.errstate(over='ignore',invalid='ignore'):
        full[:,work_idx]=np.asarray(partial,dtype=np.float32)
    if not np.isfinite(full).all(): raise ValueError('LIME float32 overflow/non-finite perturbations')
    return full


def weighted_metrics(y,p,w):
    w=np.asarray(w,dtype=float);w=w/max(w.sum(),1e-15)
    center=np.sum(w*y);var=np.sum(w*(y-center)**2)
    r2=float(1-np.sum(w*(y-p)**2)/var) if var>1e-8 else np.nan
    return r2,float(np.sum(w*np.abs(y-p))),float(var)

LIME_METRICS,LIME_FEATURE_ROWS=[],[]
LIME_NEIGHBOR_HASHES={}
LIME_DETAILS={}
for i in analysis_indices:
    case=case_table.iloc[i];query=local_X[i]
    dist=normalized_l1(REF_X,query)
    context_ids=np.flatnonzero(context_match(REF_X,query))
    eligible=context_ids if len(context_ids)>=LIME_MIN_CONTEXT_BACKGROUND else np.arange(len(REF_X))
    neighbors=eligible[np.argsort(dist[eligible],kind='stable')[:LIME_LOCAL_BACKGROUND_N]]
    background=REF_X[neighbors]
    work_idx=np.asarray([j for j in MUTABLE_IDX if np.ptp(background[:,j])>1e-8],dtype=np.int64)
    if not len(work_idx): raise RuntimeError(f'{case.case_id}: local background has no varying mutable features')
    work_names=[FEATURES[j] for j in work_idx]
    cats=[k for k,j in enumerate(work_idx) if BINARY[j]]
    np.save(OUT_DIR/(case.case_id+'_lime_background_pilot_indices.npy'),ref_idx[neighbors])
    for cfg in LIME_CONFIGS:
        for repeat_seed in LIME_SEEDS:
            case_seed=int(repeat_seed+1000+i)
            explainer=LimeTabularExplainer(background[:,work_idx],feature_names=work_names,class_names=CLASS_NAMES,
                mode='classification',categorical_features=cats,discretize_continuous=cfg['discretize_continuous'],
                sample_around_instance=True,kernel_width=np.sqrt(len(work_idx))*cfg['kernel_factor'],random_state=case_seed)
            # Pin to LIME 0.2.0.1. This helper returns interpretable data and inverse/raw samples.
            generator=getattr(explainer,'_LimeTabularExplainer__data_inverse',None)
            if generator is None: raise RuntimeError('LIME neighborhood helper missing; check pinned version.')
            explainer.random_state.seed(case_seed+100000)
            held_data,held_inverse=generator(query[work_idx],LIME_HOLDOUT_N+1)
            held_full=reconstruct_lime_input(query,held_inverse,work_idx)
            held_scaled=(held_data-explainer.scaler.mean_)/explainer.scaler.scale_
            held_distance=np.linalg.norm(held_scaled-held_scaled[0],axis=1)
            held_weights=explainer.base.kernel_fn(held_distance)[1:]
            held_valid=domain_valid(held_full,query)[1:]
            for name in MODELS:
                held_start=time.perf_counter()
                held_p=ESTIMATORS[name].predict_proba(held_full)
                held_seconds=time.perf_counter()-held_start
                held_hash=hashlib.sha256(held_full.tobytes()).hexdigest()
                hash_key=(case.case_id,cfg['config_id'],repeat_seed,'holdout')
                assert LIME_NEIGHBOR_HASHES.setdefault(hash_key,held_hash)==held_hash
                for target in CASE_TARGETS[case.case_id]:
                    explainer.random_state.seed(case_seed)
                    captured=[]
                    def predict_fn(partial,model_name=name):
                        full=reconstruct_lime_input(query,partial,work_idx)
                        captured.append((hashlib.sha256(full.tobytes()).hexdigest(),float(domain_valid(full,query).mean())))
                        return ESTIMATORS[model_name].predict_proba(full)
                    start=time.perf_counter()
                    exp=explainer.explain_instance(query[work_idx],predict_fn,labels=(target,),
                        num_features=min(LIME_NUM_FEATURES,len(work_idx)),num_samples=LIME_NUM_SAMPLES)
                    elapsed=time.perf_counter()-start
                    if len(captured)!=1: raise RuntimeError('Unexpected LIME predict_fn calls; cannot verify shared neighborhood')
                    fit_hash,fit_valid_fraction=captured[0]
                    fit_key=(case.case_id,cfg['config_id'],repeat_seed,'fit')
                    assert LIME_NEIGHBOR_HASHES.setdefault(fit_key,fit_hash)==fit_hash
                    linear=np.full(len(held_scaled),float(exp.intercept[target]))
                    for k,weight in exp.local_exp[target]: linear+=float(weight)*held_scaled[:,k]
                    r2,mae,var=weighted_metrics(held_p[1:,target],linear[1:],held_weights)
                    point_p=lime_scalar(exp.local_pred,target);model_p=float(LOCAL_PROB[name][i,target])
                    point_error=abs(point_p-model_p)
                    valid_fraction=float(held_valid.mean())
                    trusted=bool(np.isfinite(r2) and r2>=LIME_MIN_HOLDOUT_R2 and point_error<=LIME_MAX_POINT_ERROR
                                 and valid_fraction>=LIME_MIN_VALID_NEIGHBOR_FRACTION and fit_valid_fraction>=LIME_MIN_VALID_NEIGHBOR_FRACTION)
                    rec=dict(case_id=case.case_id,model=name,explained_class=CLASS_NAMES[target],class_id=target,
                        config_id=cfg['config_id'],repeat_seed=repeat_seed,perturbation_seed=case_seed,
                        train_weighted_r2=lime_scalar(exp.score,target),holdout_weighted_r2=r2,holdout_weighted_mae=mae,
                        holdout_weighted_variance=var,model_probability=model_p,lime_local_prediction=point_p,
                        abs_point_probability_error=point_error,fit_valid_fraction=fit_valid_fraction,
                        holdout_valid_fraction=valid_fraction,trusted_by_project_rules=trusted,
                        fit_neighborhood_sha256=fit_hash,holdout_neighborhood_sha256=held_hash,
                        background_policy='same_context_nearest' if len(context_ids)>=LIME_MIN_CONTEXT_BACKGROUND else 'nearest_all_contexts_conditional_query',
                        background_rows=len(neighbors),explainable_features=len(work_idx),seconds=elapsed)
                    LIME_METRICS.append(rec)
                    feature_rows=[]
                    conditions = dict(zip([k for k,_ in exp.local_exp[target]], [text for text,_ in exp.as_list(label=target)]))
                    for rank,(k,weight) in enumerate(exp.local_exp[target],1):
                        j=int(work_idx[k]);row=dict(case_id=case.case_id,model=name,explained_class=CLASS_NAMES[target],
                            config_id=cfg['config_id'],repeat_seed=repeat_seed,rank=rank,feature=FEATURES[j],
                            feature_value=float(query[j]),feature_condition=conditions[k],lime_weight=float(weight),trusted_by_project_rules=trusted)
                        feature_rows.append(row);LIME_FEATURE_ROWS.append(row)
                    LIME_DETAILS[(case.case_id,name,target,cfg['config_id'],repeat_seed)]=feature_rows
                    TIMINGS.append(dict(method='LIME_fit',model=name,case_id=case.case_id,explained_class=CLASS_NAMES[target],
                        config_id=cfg['config_id'],repeat_seed=repeat_seed,seconds=elapsed,samples=LIME_NUM_SAMPLES))
                TIMINGS.append(dict(method='LIME_holdout_prediction',model=name,case_id=case.case_id,
                    config_id=cfg['config_id'],repeat_seed=repeat_seed,seconds=held_seconds,samples=LIME_HOLDOUT_N+1))
            print(case.case_id,cfg['config_id'],repeat_seed,'LIME completed',flush=True)
    pd.DataFrame(LIME_METRICS).to_csv(OUT_DIR/'lime_fidelity_all_configs.csv',index=False)
    pd.DataFrame(LIME_FEATURE_ROWS).to_csv(OUT_DIR/'lime_features_all_configs.csv',index=False)
    pd.DataFrame(TIMINGS).to_csv(OUT_DIR/'explanation_timings.csv',index=False)


In [ ]:
lime_metrics=pd.DataFrame(LIME_METRICS)
lime_features=pd.DataFrame(LIME_FEATURE_ROWS)
selected_configs=[]
for (case_id,label),group in lime_metrics.groupby(['case_id','explained_class'],sort=False):
    candidates=[]
    for config_id,g in group.groupby('config_id',sort=False):
        r2=g.holdout_weighted_r2.to_numpy()
        # All outputs nearly constant => rank below any informative configuration.
        score=float(np.nanmean(r2)) if np.isfinite(r2).any() else -1e12
        candidates.append((score,-float(g.abs_point_probability_error.mean()),config_id))
    _,_,best=max(candidates,key=lambda x:(x[0],x[1]))
    selected_configs.append(dict(case_id=case_id,explained_class=label,config_id=best))
selected_configs=pd.DataFrame(selected_configs)
selected_metrics=lime_metrics.merge(selected_configs,on=['case_id','explained_class','config_id'])
selected_features=lime_features.merge(selected_configs,on=['case_id','explained_class','config_id'])
selected_configs.to_csv(OUT_DIR/'lime_common_selected_configs.csv',index=False)
selected_metrics.to_csv(OUT_DIR/'lime_selected_fidelity.csv',index=False)
selected_features.to_csv(OUT_DIR/'lime_selected_features.csv',index=False)

stability=[]
for (case_id,name,label,config_id),group in selected_features.groupby(['case_id','model','explained_class','config_id'],sort=False):
    sets={seed:set(g.loc[g.lime_weight.abs()>1e-12,'feature']) for seed,g in group.groupby('repeat_seed')}
    for a,b in combinations(sorted(sets),2):
        union=sets[a]|sets[b]
        stability.append(dict(case_id=case_id,model=name,explained_class=label,config_id=config_id,
            seed_a=int(a),seed_b=int(b),top_features_jaccard=len(sets[a]&sets[b])/len(union) if union else np.nan))
pd.DataFrame(stability).to_csv(OUT_DIR/'lime_seed_stability.csv',index=False)

# ------------------ Visualize giải thích LIME chuẩn format ------------------
import matplotlib.patches as mpatches

plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

for i in analysis_indices:
    case = case_table.iloc[i]
    target = int(case.y_true_id)
    cfg_id = selected_configs.loc[(selected_configs.case_id == case.case_id) & (selected_configs.explained_class == case.y_true)].iloc[0].config_id
    repeat_seed = LIME_SEEDS[0]
    
    fig, axes = plt.subplots(1, 3, figsize=(19, 6.5), dpi=140)
    
    for ax, name in zip(axes, MODELS):
        rows = LIME_DETAILS[(case.case_id, name, target, cfg_id, repeat_seed)]
        rows = sorted(rows, key=lambda r: abs(r['lime_weight']))
        
        conditions = [r['feature_condition'] for r in rows]
        weights = [r['lime_weight'] for r in rows]
        colors = ['#E74C3C' if w > 0 else '#2980B9' for w in weights]
        
        bars = ax.barh(conditions, weights, color=colors, height=0.65, edgecolor='white', zorder=3)
        ax.axvline(0, color='#333333', linewidth=0.8, linestyle='--')
        
        # Ghi giá trị số bên cạnh thanh bar
        val_range = max([abs(w) for w in weights] + [1e-4])
        for bar, w in zip(bars, weights):
            ha = 'left' if w >= 0 else 'right'
            offset = val_range * 0.02 if w >= 0 else -val_range * 0.02
            ax.text(w + offset, bar.get_y() + bar.get_height()/2, f"{w:+.3f}",
                    va='center', ha=ha, fontsize=8, color='#111111', fontweight='bold')
            
        ax.set_xlim(-val_range * 1.35, val_range * 1.35)
        ax.set_xlabel('Hệ số mô hình đại diện LIME (Surrogate Coefficient)', fontsize=10, fontweight='bold', labelpad=6)
        
        q = selected_metrics[(selected_metrics.case_id == case.case_id) & (selected_metrics.model == name) &
                             (selected_metrics.class_id == target) & (selected_metrics.repeat_seed == repeat_seed)].iloc[0]
        
        pred_class = CLASS_NAMES[int(LOCAL_PRED[name][i])]
        is_correct = (int(LOCAL_PRED[name][i]) == target)
        tag = "✓ ĐÚNG" if is_correct else "✗ NHẦM"
        title_color = "#27AE60" if is_correct else "#C0392B"
        
        ax.set_title(f"{name} [{tag}] — Dự đoán: {pred_class}\n"
                     f"R² holdout = {q.holdout_weighted_r2:.2f} | Sai số điểm = {q.abs_point_probability_error:.2f}",
                     fontsize=11, fontweight='bold', pad=10, color=title_color)
        
        if not q.trusted_by_project_rules:
            ax.text(0.98, 0.04, '⚠ LOW FIDELITY / KHÔNG ĐỦ TIN CẬY', transform=ax.transAxes,
                    color='#C0392B', fontsize=8, fontweight='bold', ha='right',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='#FDEDEC', edgecolor='#E74C3C', alpha=0.9))
            
        ax.grid(axis='x', linestyle=':', linewidth=0.5, alpha=0.6, color='#BBBBBB', zorder=0)
        ax.set_facecolor('#FAFAFA')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.tick_params(labelsize=8.5)

    pos_patch = mpatches.Patch(color='#E74C3C', label='Tăng khả năng phân loại vào lớp (+)')
    neg_patch = mpatches.Patch(color='#2980B9', label='Giảm khả năng phân loại vào lớp (-)')
    fig.legend(handles=[pos_patch, neg_patch], loc='lower center', ncol=2, frameon=True,
               facecolor='white', edgecolor='#E0E0E0', fontsize=10, bbox_to_anchor=(0.5, -0.03))

    fig.suptitle(
        f"CICIoT2023 — Giải thích cục bộ bằng LIME (Surrogate Explanations) | Ca: {case.case_id}\n"
        f"Lớp thực tế & Lớp phân tích: {case.y_true} | Cấu hình LIME: {cfg_id}",
        fontsize=13.5, fontweight='bold', y=1.03, color='#111111'
    )
    fig.tight_layout()
    fig.savefig(OUT_DIR / (case.case_id + '_lime_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
display(selected_metrics[['case_id','model','explained_class','config_id','repeat_seed','holdout_weighted_r2',
                          'abs_point_probability_error','holdout_valid_fraction','trusted_by_project_rules']])


## 7. Counterfactual: prototype gần nhất + greedy restore theo nhóm

Mục tiêu chung của một ca: nếu có model dự đoán sai, dùng lớp thật làm lớp đích cho cả ba; model đã đúng được ghi `already_target` và không tính là thành công tìm CF. Nếu cả ba đúng, dùng Normal làm lớp tương phản cho ca tấn công; với Normal, chọn lớp tấn công có xác suất trung bình cao nhất. Đây là lựa chọn what-if cố định và được lưu trước tìm kiếm.

Tìm prototype **train** có nhãn thật bằng lớp đích, được model dự đoán đúng với probability ≥0,5 và không hòa top-1. Giữ nguyên context. Xếp theo `mean normalized L1 + 0.1 × changed_feature_fraction`; thử hoàn trả nhóm feature về query miễn vẫn thỏa constraints và đạt lớp đích. Không hạ threshold/nới context khi thất bại.

Lưu prototype quan sát và CF cuối. Nếu là hybrid sau restore, đánh dấu synthetic; nhãn thật của hybrid **không biết**, chỉ có prediction của model. Sự đổi lớp sau thay feature là giả định của mô hình, không phải chứng minh nhân quả hay hướng dẫn thay lưu lượng thật.


In [ ]:
CF_TASKS=[]
for i in analysis_indices:
    case=case_table.iloc[i];true=int(case.y_true_id)
    if any(LOCAL_PRED[n][i] != true for n in MODELS):
        target=true;purpose='recover_true_class_for_misclassified_case'
    elif true != 0:
        target=0;purpose='contrast_attack_to_Normal_for_correct_case'
    else:
        mean_p=np.mean([LOCAL_PROB[n][i] for n in MODELS],axis=0).copy();mean_p[true]=-1
        target=int(mean_p.argmax());purpose='contrast_Normal_to_attack_for_correct_case'
    CF_TASKS.append(dict(case_id=case.case_id,local_position=int(i),y_true_id=true,target_id=target,
                         target_class=CLASS_NAMES[target],purpose=purpose))
pd.DataFrame(CF_TASKS).to_csv(OUT_DIR/'counterfactual_tasks.csv',index=False)

group_names={
    'duration_timing':['flow_duration','Duration','IAT'],
    'rates':['Rate','Srate','Drate'],
    'flags':['fin_flag_number','syn_flag_number','rst_flag_number','psh_flag_number','ack_flag_number','ece_flag_number','cwr_flag_number'],
    'flag_counts':['ack_count','syn_count','fin_count','urg_count','rst_count'],
    'size_statistics':['Tot sum','Min','Max','AVG','Std','Tot size','Magnitue','Radius','Covariance','Variance'],
    'count_weight':['Number','Weight'],
    'header':['Header_Length'],
}
CF_GROUPS=[]
assigned=set()
for group,features in group_names.items():
    ids=[FEATURES.index(f) for f in features if f in FEATURES and FEATURES.index(f) in set(MUTABLE_IDX)]
    if ids:CF_GROUPS.append((group,np.asarray(ids,dtype=np.int64)));assigned.update(ids)
for j in MUTABLE_IDX:
    if j not in assigned:CF_GROUPS.append((FEATURES[j],np.asarray([j],dtype=np.int64)))
write_json(OUT_DIR/'counterfactual_groups.json',{g:[FEATURES[j] for j in ids] for g,ids in CF_GROUPS})


def changed_mask(x,query):
    return ~np.isclose(x,query,rtol=CF_CHANGE_RTOL,atol=CF_CHANGE_ATOL)


def cf_objective(x,query):
    distance=float(normalized_l1(x,query))
    n=int(changed_mask(x,query).sum())
    return distance+CF_SPARSITY_WEIGHT*n/max(len(MUTABLE_IDX),1),distance,n


def target_valid(prob,target):
    other=np.delete(prob,target,axis=1).max(axis=1)
    return (prob.argmax(axis=1)==target)&(prob[:,target]>=CF_MIN_TARGET_PROB)&(prob[:,target]-other>CF_MIN_TARGET_GAP)


def refine_counterfactual(name,query,seed,target):
    current=seed.copy();history=[]
    for round_id in range(CF_RESTORE_ROUNDS):
        candidates=[];groups=[]
        for group,ids in CF_GROUPS:
            if not changed_mask(current[ids],query[ids]).any():continue
            candidate=current.copy();candidate[ids]=query[ids]
            if domain_valid(candidate,query,strict_reference_bounds=True)[0]:
                candidates.append(candidate);groups.append(group)
        if not candidates:break
        matrix=np.asarray(candidates,dtype=np.float32)
        probabilities=ESTIMATORS[name].predict_proba(matrix)
        valid=target_valid(probabilities,target)
        current_score=cf_objective(current,query)[0]
        options=[(cf_objective(matrix[j],query)[0],j) for j in np.flatnonzero(valid)]
        if not options:break
        score,j=min(options,key=lambda t:t[0])
        if score>=current_score-1e-12:break
        current=matrix[j].copy();history.append(groups[j])
    return current,history

CF_DETAIL_COLUMNS = ['case_id','model','cf_id','target_class','target_probability',
    'normalized_l1','changed_features','objective','synthetic_hybrid',
    'source_pilot_row_index','source_train_row_index','restored_groups','traffic_feasibility']
CF_VECTOR_COLUMNS = CF_DETAIL_COLUMNS + FEATURES
CF_CHANGE_COLUMNS = CF_DETAIL_COLUMNS + ['feature','before','after','delta','normalized_change']
CF_PROTOTYPE_COLUMNS = ['cf_id'] + FEATURES

CF_SUMMARY,CF_CHANGES,CF_VECTORS,CF_PROTOTYPES=[],[],[],[]
for task in CF_TASKS:
    i=task['local_position'];query=local_X[i];target=task['target_id']
    for name in MODELS:
        start=time.perf_counter();base=dict(case_id=task['case_id'],model=name,target_class=CLASS_NAMES[target],target_id=target,
            y_true=CLASS_NAMES[int(local_true[i])],original_prediction=CLASS_NAMES[int(LOCAL_PRED[name][i])],
            purpose=task['purpose'],original_target_probability=float(LOCAL_PROB[name][i,target]))
        if int(LOCAL_PRED[name][i])==target:
            CF_SUMMARY.append(dict(**base,status='already_target',search_eligible=False,valid_cfs=0,seconds=0.0))
            continue
        donor_mask=(REF_Y==target)&target_valid(REF_PROB[name],target)
        candidates=np.flatnonzero(donor_mask)
        status='no_target_donor_in_reference'
        if len(candidates):
            candidates=candidates[context_match(REF_X[candidates],query)]
            status='no_donor_matching_fixed_context'
        if len(candidates):
            candidates=candidates[domain_valid(REF_X[candidates],query,strict_reference_bounds=True)]
            status='no_domain_valid_donor'
        results=[]
        if len(candidates):
            ranked=sorted(candidates,key=lambda k:cf_objective(REF_X[k],query)[0])[:CF_SEED_DONORS]
            for donor in ranked:
                final,history=refine_counterfactual(name,query,REF_X[donor],target)
                prob=PIPELINES[name].predict_proba(raw_frame(final[None,:]))[0]
                assert np.allclose(prob,ESTIMATORS[name].predict_proba(final[None,:])[0],rtol=1e-6,atol=1e-7)
                assert domain_valid(final,query,strict_reference_bounds=True)[0]
                assert target_valid(prob[None,:],target)[0]
                score,distance,nchanged=cf_objective(final,query)
                if CF_MAX_CHANGED_FEATURES is not None and nchanged>CF_MAX_CHANGED_FEATURES:continue
                results.append(dict(vector=final,prob=prob,donor=int(donor),restored=history,score=score,distance=distance,nchanged=nchanged))
            results.sort(key=lambda x:x['score'])
            unique=[];seen=set()
            for result in results:
                key=result['vector'].tobytes()
                if key not in seen:unique.append(result);seen.add(key)
            results=unique[:CF_TOTAL]
            status='success' if results else 'no_cf_meeting_change_limit'
        elapsed=time.perf_counter()-start
        CF_SUMMARY.append(dict(**base,status=status,search_eligible=True,valid_cfs=len(results),
            best_normalized_l1=results[0]['distance'] if results else np.nan,
            best_changed_features=results[0]['nchanged'] if results else np.nan,seconds=elapsed))
        for k,result in enumerate(results,1):
            cf_id=f"{task['case_id']}_{name}_cf{k}"
            donor=result['donor'];final=result['vector'];synthetic=not np.array_equal(final,REF_X[donor])
            details=dict(case_id=task['case_id'],model=name,cf_id=cf_id,target_class=CLASS_NAMES[target],
                target_probability=float(result['prob'][target]),normalized_l1=result['distance'],
                changed_features=result['nchanged'],objective=result['score'],synthetic_hybrid=synthetic,
                source_pilot_row_index=int(ref_idx[donor]),source_train_row_index=int(REF_SOURCE_IDS[donor]),
                restored_groups=';'.join(result['restored']),traffic_feasibility='not_certified')
            CF_VECTORS.append(dict(**details,**dict(zip(FEATURES,map(float,final)))))
            CF_PROTOTYPES.append(dict(cf_id=cf_id,**dict(zip(FEATURES,map(float,REF_X[donor])))))
            for j in np.flatnonzero(changed_mask(final,query)):
                CF_CHANGES.append(dict(**details,feature=FEATURES[j],before=float(query[j]),after=float(final[j]),
                    delta=float(final[j]-query[j]),normalized_change=float(abs(distance_encode(final)[j]-distance_encode(query)[j])/DIST_SCALE[j])))
        TIMINGS.append(dict(method='counterfactual_search',model=name,case_id=task['case_id'],seconds=elapsed,samples=len(results)))
        print(task['case_id'],name,'target',CLASS_NAMES[target],status,flush=True)
    pd.DataFrame(CF_SUMMARY).to_csv(OUT_DIR/'counterfactual_summary.csv',index=False)
    pd.DataFrame(CF_VECTORS,columns=CF_VECTOR_COLUMNS).to_csv(OUT_DIR/'counterfactual_vectors.csv',index=False)
    pd.DataFrame(CF_CHANGES,columns=CF_CHANGE_COLUMNS).to_csv(OUT_DIR/'counterfactual_feature_changes.csv',index=False)
    pd.DataFrame(CF_PROTOTYPES,columns=CF_PROTOTYPE_COLUMNS).to_csv(OUT_DIR/'counterfactual_observed_prototypes.csv',index=False)
    pd.DataFrame(TIMINGS).to_csv(OUT_DIR/'explanation_timings.csv',index=False)


In [ ]:
cf_summary=pd.DataFrame(CF_SUMMARY)
cf_changes=pd.DataFrame(CF_CHANGES,columns=CF_CHANGE_COLUMNS)
cf_vectors=pd.DataFrame(CF_VECTORS,columns=CF_VECTOR_COLUMNS)
cf_quality=[]
for name in MODELS:
    t=cf_summary[cf_summary.model==name]
    eligible=t[t.search_eligible]
    successful=eligible[eligible.status=='success']
    cf_quality.append(dict(model=name,all_tasks=len(t),already_target=int((t.status=='already_target').sum()),
        searches=len(eligible),successful_searches=len(successful),
        success_rate_on_searched_cases=len(successful)/len(eligible) if len(eligible) else np.nan,
        mean_best_changed_features=successful.best_changed_features.mean() if len(successful) else np.nan,
        mean_best_normalized_l1=successful.best_normalized_l1.mean() if len(successful) else np.nan,
        total_search_seconds=eligible.seconds.sum()))
    t.to_csv(OUT_DIR/name/'counterfactual_summary.csv',index=False)
pd.DataFrame(cf_quality).to_csv(OUT_DIR/'counterfactual_model_comparison.csv',index=False)
display(pd.DataFrame(cf_quality))

# ------------------ Visualize Counterfactual chuẩn format ------------------
for task in CF_TASKS:
    fig, axes = plt.subplots(1, 3, figsize=(19, 6.2), dpi=140)
    for ax, name in zip(axes, MODELS):
        key = f"{task['case_id']}_{name}_cf1"
        rows = cf_changes[cf_changes.cf_id == key] if not cf_changes.empty else pd.DataFrame()
        if rows.empty:
            status = cf_summary[(cf_summary.case_id == task['case_id']) & (cf_summary.model == name)].iloc[0].status
            if status == 'already_target':
                msg = "✓ ĐÃ THUỘC LỚP MỤC TIÊU\n(Mô hình ban đầu đã dự đoán đúng,\nkhông cần can thiệp đặc trưng)"
                box_color = '#E8F8F5'
                text_color = '#1E8449'
                border_color = '#2ECC71'
            else:
                msg = f"✕ KHÔNG TÌM THẤY NGHỊCH ĐẢO\n(Trạng thái: {status})"
                box_color = '#F2F4F4'
                text_color = '#7F8C8D'
                border_color = '#BDC3C7'
                
            ax.set_facecolor('#FAFAFA')
            ax.text(0.5, 0.5, msg, ha='center', va='center', transform=ax.transAxes,
                    fontsize=10.5, fontweight='bold', color=text_color,
                    bbox=dict(boxstyle='round,pad=0.8', facecolor=box_color, edgecolor=border_color, linewidth=1.2))
            ax.set_xticks([])
            ax.set_yticks([])
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.spines['left'].set_visible(False)
            ax.spines['bottom'].set_visible(False)
            ax.set_title(f"{name}\n(Không có can thiệp)", fontsize=11.5, fontweight='bold', pad=10, color='#555555')
        else:
            rows = rows.nlargest(12, 'normalized_change').sort_values('normalized_change')
            bars = ax.barh(rows.feature, rows.normalized_change, color='#2980B9', height=0.65, edgecolor='white', zorder=3)
            
            # Ghi nhãn giá trị độ thay đổi chuẩn hóa
            max_c = rows.normalized_change.max() if len(rows) else 1.0
            for bar in bars:
                w = bar.get_width()
                ax.text(w + max_c * 0.02, bar.get_y() + bar.get_height()/2, f"{w:.3f}",
                        va='center', ha='left', fontsize=8.5, color='#2C3E50', fontweight='bold')
                
            ax.set_xlim(0, max_c * 1.25)
            ax.set_xlabel('Độ dịch chuyển chuẩn hóa (|Δx| / IQR)', fontsize=10, fontweight='bold', labelpad=6)
            ax.set_title(f"{name}\n(Đổi {len(rows)} đặc trưng quan trọng)", fontsize=11.5, fontweight='bold', pad=10, color='#1A252C')
            ax.grid(axis='x', linestyle=':', linewidth=0.5, alpha=0.6, color='#BBBBBB', zorder=0)
            ax.set_facecolor('#FAFAFA')
            ax.spines['top'].set_visible(False)
            ax.spines['right'].set_visible(False)
            ax.tick_params(labelsize=8.5)

    fig.suptitle(
        f"CICIoT2023 — Phân tích điều kiện nghịch đảo (Counterfactual Explanations) | Ca: {task['case_id']}\n"
        f"Lớp mục tiêu chuyển đổi (Target Class): {task['target_class']} ({task['purpose']})",
        fontsize=13.5, fontweight='bold', y=1.03, color='#111111'
    )
    fig.tight_layout()
    fig.savefig(OUT_DIR / (task['case_id'] + '_counterfactual_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 8. Tổng hợp và bàn giao

Ghi cấu hình/seed/hashes, kết quả quality của cả hai phương pháp và mọi thất bại CF. `already_target` được loại khỏi mẫu số success rate. LIME có nhãn quality thấp vẫn được lưu để báo cáo trung thực, không tự coi là hoàn thành chất lượng nghiên cứu chỉ vì notebook chạy hết.


In [ ]:
timings=pd.DataFrame(TIMINGS)
timings.to_csv(OUT_DIR/'explanation_timings.csv',index=False)
timing_summary=timings.groupby(['model','method'],sort=False).agg(calls=('seconds','size'),total_seconds=('seconds','sum')).reset_index()
timing_summary.to_csv(OUT_DIR/'explanation_timing_summary.csv',index=False)
quality_summary=selected_metrics.groupby('model',sort=False).agg(
    explanations=('case_id','size'),mean_holdout_r2=('holdout_weighted_r2','mean'),
    mean_point_error=('abs_point_probability_error','mean'),
    mean_valid_neighbor_fraction=('holdout_valid_fraction','mean'),
    trusted_fraction=('trusted_by_project_rules','mean')).reset_index()
quality_summary.to_csv(OUT_DIR/'lime_model_comparison.csv',index=False)
for name in MODELS:
    selected_metrics[selected_metrics.model==name].to_csv(OUT_DIR/name/'lime_fidelity.csv',index=False)
    selected_features[selected_features.model==name].to_csv(OUT_DIR/name/'lime_explanations.csv',index=False)
    cf_changes[cf_changes.model==name].to_csv(OUT_DIR/name/'counterfactual_feature_changes.csv',index=False)
    cf_vectors[cf_vectors.model==name].to_csv(OUT_DIR/name/'counterfactual_vectors.csv',index=False)

display(quality_summary);display(timing_summary)
manifest=dict(run_id=RUN_ID,methods=['LIME','counterfactual'],baseline_run_id=EXPECTED_MODEL_RUN,
    source_run_id=EXPECTED_PROCESSING_RUN,pilot_run_id=EXPECTED_PILOT_RUN,
    paper_reference=dict(doi='10.1109/ACCESS.2023.3336678',implementation='inspired methodology; separate baseline models; not exact reproduction'),
    model_metadata={n:dict(experiment_id=CONFIGS[n]['experiment_id'],pipeline_sha256=BUNDLES[n]['pipeline_sha256'],
                           model_sha256=BUNDLES[n]['model_sha256'],golden_check='passed') for n in MODELS},
    class_order=CLASS_NAMES,feature_order=FEATURES,seed=SEED,
    evaluation_policy='audited validation; baseline validation/test previously viewed; targeted exploratory cases',
    shared_case_ids=case_table.iloc[analysis_indices].case_id.tolist(),
    fixed_context=FIXED_CONTEXT_FEATURES,reference_pool_rows=len(REF_X),reference_per_class_max=TRAIN_REFERENCE_PER_CLASS,
    lime=dict(configs=LIME_CONFIGS,seeds=LIME_SEEDS,num_samples=LIME_NUM_SAMPLES,holdout_samples=LIME_HOLDOUT_N,
        local_background_max=LIME_LOCAL_BACKGROUND_N,conditional_on_fixed_context=True,
        quality_rules=dict(min_holdout_r2=LIME_MIN_HOLDOUT_R2,max_point_error=LIME_MAX_POINT_ERROR,
                           min_valid_neighbor_fraction=LIME_MIN_VALID_NEIGHBOR_FRACTION),
        config_selection='one common config per case/class by mean holdout R2 across models/seeds; descriptive, not unbiased final evaluation',
        neighborhood_matching='SHA256 equality across models/target classes verified'),
    counterfactual=dict(method='train prototype retrieval + greedy grouped restoration',total_CFs=CF_TOTAL,
        seed_donors=CF_SEED_DONORS,restore_rounds=CF_RESTORE_ROUNDS,min_target_probability=CF_MIN_TARGET_PROB,
        min_target_gap=CF_MIN_TARGET_GAP,max_changed_features=CF_MAX_CHANGED_FEATURES,
        distance='mean robust-scaled log1p/raw L1; train reference statistics only',
        sparsity_weight=CF_SPARSITY_WEIGHT,change_rtol=CF_CHANGE_RTOL,change_atol=CF_CHANGE_ATOL,
        optimality='not guaranteed; finite prototype search',traffic_feasibility='not certified',
        success_denominator='searched cases only; exclude already_target'),
    verify_input_hashes=VERIFY_INPUT_HASHES,metadata_loaded=VAL_METADATA is not None,
    runtime_versions={v:package_metadata.version(v) for v in ['numpy','pandas','scikit-learn','joblib','xgboost','lime']},
    hardware=dict(platform=platform.platform(),cpu_count=os.cpu_count(),processor=platform.processor(),
        python=platform.python_version(),model_threads={n:CONFIGS[n]['model_params'].get('n_jobs') for n in MODELS}))
write_json(OUT_DIR/'xai_run_manifest.json',manifest)
write_json(OUT_DIR/'run_status.json',dict(run_id=RUN_ID,status='execution_complete',models=MODELS,
    lime=True,counterfactual=True,research_quality='inspect fidelity/domain/stability and CF successes; not automatically accepted'))
print('Saved:',OUT_DIR)
for f in sorted(OUT_DIR.iterdir()):print(' -',f.name)


## 9. Đọc kết quả cho báo cáo

- `shared_analysis_cases.csv`, `local_predictions_three_models.csv`: cùng query, class order và dự đoán từng model. Một sample có thể đúng ở model này nhưng sai ở model khác.
- `lime_fidelity_all_configs.csv`: toàn bộ config/seed, train và holdout fidelity; `lime_selected_fidelity.csv` là cấu hình chung được chọn. Nhãn quality thấp cần được trình bày như giới hạn, không che đi.
- `lime_seed_stability.csv`: độ ổn định top features qua seed. R² holdout cao nhưng feature thay đổi mạnh vẫn cần thảo luận.
- Các ảnh `case_*_lime_comparison.png`: dấu hệ số surrogate có điều kiện trên context cố định; không phải SHAP và không phải tác động nhân quả.
- `counterfactual_summary.csv`: success/failure/already_target; `counterfactual_vectors.csv` là các feature vectors sau thay đổi, `counterfactual_feature_changes.csv` ghi trước/sau. `counterfactual_observed_prototypes.csv` lưu mẫu gốc được dùng làm seed.
- CF synthetic chỉ có nhãn dự đoán; không tự gán nhãn thật bằng target. Một CF thỏa model/constraints không chứng minh thay đổi đó có thể thực hiện trên lưu lượng mạng. `no_*` nghĩa là chưa tìm thấy trong pool/ràng buộc/giới hạn chạy hiện tại, không chứng minh không tồn tại CF.
- `explanation_timings.csv`: tách fit LIME, prediction holdout, reference prediction và CF search; không so tốc độ hai phương pháp chỉ bằng tổng thời gian.

LIME + counterfactual giúp phân tích quyết định; không làm tăng recall/precision của estimator. Chỉ kết luận về chất lượng XAI sau khi đọc fidelity, stability, validity và số ca. Các ngưỡng quality là quy ước thực nghiệm cần công bố. Phân tích thêm nhiều case/seed và xác minh trên dữ liệu mới nếu muốn kết luận rộng hơn.

Nguồn: [bài báo](https://doi.org/10.1109/ACCESS.2023.3336678), [LIME API](https://lime-ml.readthedocs.io/en/latest/lime.html), [LIME source](https://github.com/marcotcr/lime/blob/master/lime/lime_tabular.py). [DiCE model-agnostic CFs](https://interpret.ml/DiCE/notebooks/DiCE_model_agnostic_CFs.html) là tham khảo về nhóm phương pháp retrieval/genetic/random; notebook này dùng thuật toán prototype + restore mô tả riêng, không dùng DiCE.
